# Introduction

When working with Large Language Models (LLMs) we often tend to use datasets composed by natural language text written with the latin alphabet. This has proven effective but there are some subtelties hidden in such a choice. The english language has many heteronyms, as in words that are spelled the same but have different meaning and pronounciation. Also the phonetics of a word can help relate it to other words or concepts even if their spelling could be not much alike.

The International Phonetics Alphabet (IPA) provides an extended alphabet that precisely describes how words are pronounced. The usage of such an extended alphabet, instead of the latin alphabet, could help LLMs to improve there understanding of the dataset and consequently their accuracy in inference.

The project takes a well known dataset, the imdb movie reviews, and generates an analogous IPA dataset by using the [phonemizer](https://pypi.org/project/phonemizer/) library.

Then it uses the original and IPA datasets to train two models and then compares the results.

# Setup

In [1]:
!pip install transformers[torch] phonemizer torch pandas
!pip install accelerate -U
!apt-get install -y espeak

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 955.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.9/566.9 kB 48.2 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-data libespeak1 libportaudio2 libsonic0
The following NEW packages will be installed:
  espeak espeak-data libespeak1 libportaudio2 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,382 kB of archives.
After this operation, 3,178 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/univers

# Imports and Dataset Load

In [2]:
from urllib.request import urlretrieve

def download(file, url):
    if not os.path.isfile(file):
        urlretrieve(url, file)

def strip_tags(text):
    return text.replace("<br />", "\n")

In [3]:
import os
import pandas as pd


download("imdb-train.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-train.csv.gz")
train_set = pd.read_csv("imdb-train.csv.gz", sep="\t", names=["label", "text"])
train_set["text"] = train_set["text"].apply(strip_tags)
download("imdb-test.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-test.csv.gz")
test_set = pd.read_csv("imdb-test.csv.gz", sep="\t", names=["label", "text"])
test_set["text"] = test_set["text"].apply(strip_tags)


### Dataset preparation

In [4]:
from sklearn import preprocessing
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from sklearn.model_selection import train_test_split
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def encode_labels(labels):
    le = preprocessing.LabelEncoder()
    return le.fit_transform(labels)

def prepare_dataset(tokenizer, dataset):
    labels = encode_labels(dataset["label"])
    texts = dataset["text"]
    if type(texts) != list:
        texts = texts.tolist()

    encodings = tokenizer(texts, truncation=True, padding=True, max_length=256)
    dataset = IMDbDataset(encodings, labels)
    return dataset

def prepare_datasets(tokenizer_params, full_train_set, test_set):
    train_set = {}
    val_set = {}
    tokenizer = AutoTokenizer.from_pretrained(**tokenizer_params)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token # manually set pad_token for gpt2 model
    train_set["text"], val_set["text"], train_set["label"], val_set["label"] = train_test_split(full_train_set["text"].tolist(), full_train_set["label"].tolist(), test_size=.1)
    train_dataset = prepare_dataset(tokenizer, train_set)
    val_dataset = prepare_dataset(tokenizer, val_set)
    test_dataset = prepare_dataset(tokenizer, test_set)
    return train_dataset, val_dataset, test_dataset, tokenizer.pad_token_id


/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or


# Phonemize Text (Convert to IPA)

In [5]:
from urllib.request import urlretrieve
from phonemizer import phonemize
import pandas as pd
import re

def phonemize_batch(batch):
    # split each text into parts (keeping track of structure)
    structured_parts = []
    all_text_parts = []

    for text in batch:
        parts = re.split(r'(<br\s*/?>)', text, flags=re.IGNORECASE)
        current_structure = []
        for part in parts:
            if re.match(r'<br\s*/?>', part, flags=re.IGNORECASE) or part.strip() == "":
                # keep HTML and empty parts as-is
                current_structure.append((part, False))
            else:
                # mark normal text part for phonemization
                current_structure.append((len(all_text_parts), True))
                all_text_parts.append(part)
        structured_parts.append(current_structure)

    # phonemize all text parts at once
    phonemized_all = phonemize(
        all_text_parts,
        language='en-us',
        backend='espeak',
        strip=False,
        preserve_punctuation=True
    )

    # reconstruct the original texts efficiently
    result = []
    for structure in structured_parts:
        reconstructed = []
        for part, is_text in structure:
            if is_text:
                reconstructed.append(phonemized_all[part])
            else:
                reconstructed.append(part)
        result.append("".join(reconstructed))
    return result

def save_phonemized_data(ipa_data, filename):
    ipa_data.to_csv(filename, index=False, encoding='utf-8')

def generate_ipa_dataset():
    download("imdb-train.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-train.csv.gz")
    train_set = pd.read_csv("imdb-train.csv.gz", sep="\t", names=["label", "text"])
    download("imdb-test.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-test.csv.gz")
    test_set = pd.read_csv("imdb-test.csv.gz", sep="\t", names=["label", "text"])

    # Phonemize dataset
    train_set["text"] = phonemize_batch(train_set["text"])
    save_phonemized_data(train_set, "ipa_dataset/ipa_train.csv")
    test_set["text"] = phonemize_batch(test_set["text"])
    save_phonemized_data(test_set, "ipa_dataset/ipa_test.csv")

Since the generation of the IPA dataset is very time consuming, the dataset has already been generated and made available through github.

In [6]:
import requests
import os
import json
import gc
import pandas as pd

def load_phonemized_data(filename):
    return pd.read_csv(filename, encoding='utf-8')

train_url = "https://raw.githubusercontent.com/Oldranda1414/ipa_bert_test/refs/heads/main/imdb_ipa_dataset/ipa_train.csv"
test_url = "https://raw.githubusercontent.com/Oldranda1414/ipa_bert_test/refs/heads/main/imdb_ipa_dataset/ipa_test.csv"

for url in [train_url, test_url]:
    filename = os.path.basename(url)
    download(filename, url)

ipa_train_set = load_phonemized_data("ipa_train.csv")
ipa_test_set = load_phonemized_data("ipa_test.csv")

ipa_train_set["text"] = ipa_train_set["text"].apply(strip_tags)
ipa_test_set["text"] = ipa_test_set["text"].apply(strip_tags)


The Tokenizer for the IPA model expects input to be space separated characters with WORD_BOUNDARY between words. The following function preprocesses the generated IPA dataset further to allign it with the ipa tokenizer's expectations:

In [7]:
def prepare_ipa_text(text, known_phonemes=None):
    """
    Split the IPA text into phoneme tokens based on the tokenizer's vocabulary
    rather than individual characters.
    """
    if known_phonemes is None:
        raise ValueError("Must pass known_phonemes from tokenizer vocab")

    # Remove all punctuation (since tokenizer doesn't support it)
    text = re.sub(r'[!"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~]', '', text)

    # Convert to lowercase (since tokenizer only has lowercase)
    text = text.lower()

    # Replace multiple spaces with single spaces
    text = re.sub(r'\s+', ' ', text)

    # Sort phonemes by descending length so that longer matches (like 'tʰ' or 'ɑː') are matched first
    sorted_phonemes = sorted(known_phonemes, key=len, reverse=True)

    # Build regex to match any known phoneme
    phoneme_pattern = re.compile("|".join(map(re.escape, sorted_phonemes)))

    processed_sentences = []
    for word in text.strip().split():
        # Find all phonemes in this word
        phonemes = phoneme_pattern.findall(word)
        if not phonemes:
            # fallback: split into characters if nothing matches (rare)
            phonemes = list(word)
        processed_sentences.append(" ".join(phonemes))

    return " WORD_BOUNDARY ".join(processed_sentences)

# Training Function

In [8]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
import torch

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def train_model(model_params, train_dataset, val_dataset, test_dataset, pad_token_id=None):
    model = AutoModelForSequenceClassification.from_pretrained(**model_params, num_labels=2)
    if pad_token_id is not None:
        model.config.pad_token_id = pad_token_id

    args = TrainingArguments(
        output_dir=f"./results-{model_params["pretrained_model_name_or_path"]}",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
        do_eval=True,
        eval_steps=200,
        report_to="none",
        optim="adamw_torch", # To solved fused=True error on TPU
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate(test_dataset)
    return metrics


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


# Train models

In [9]:
import gc
model_name = "distilbert-base-uncased"
#train_dataset, val_dataset, test_dataset, _ = prepare_datasets({"pretrained_model_name_or_path":model_name}, train_set, test_set)
#print("Training baseline model...")
#metrics_base = train_model({"pretrained_model_name_or_path":model_name}, train_dataset, val_dataset, test_dataset)
#print("Baseline:", metrics_base)


In [10]:
import gc
ipa_model_name = {"pretrained_model_name_or_path":'phonemetransformers/ipa-childes-models-tiny', "subfolder":'EnglishNA'}
ipa_tokenizer_name = {"pretrained_model_name_or_path":'phonemetransformers/ipa-childes-tokenizers', "subfolder":'EnglishUK'}

tokenizer = AutoTokenizer.from_pretrained(**ipa_tokenizer_name)

known_phonemes = [p for p in tokenizer.get_vocab().keys() if p not in {"WORD_BOUNDARY", "UTT_BOUNDARY", "PAD", "UNK"}]
for data_set in [ipa_train_set, ipa_test_set]:
    data_set["text"] = data_set["text"].apply(lambda x: prepare_ipa_text(x, known_phonemes))
ipa_train_dataset, ipa_val_dataset, ipa_test_dataset, pad_token_id = prepare_datasets(ipa_tokenizer_name, ipa_train_set, ipa_test_set)
print("Training ipa model...")
metrics_ipa = train_model(ipa_model_name, ipa_train_dataset, ipa_val_dataset, ipa_test_dataset, pad_token_id)
print("Baseline:", metrics_ipa)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/982 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

EnglishNA/model.safetensors:   0%|          | 0.00/2.54M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at phonemetransformers/ipa-childes-models-tiny and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.736600
20,0.722200
30,0.687900
40,0.710100
50,0.737600
60,0.733100
70,0.715000
80,0.718300
90,0.694600
100,0.693700


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Baseline: {'eval_loss': 0.6931000351905823, 'eval_accuracy': 0.5046, 'eval_f1': 0.6515292197743451, 'eval_precision': 0.502495551408359, 'eval_recall': 0.92624, 'eval_runtime': 36.3439, 'eval_samples_per_second': 172.133, 'eval_steps_per_second': 2.696, 'epoch': 1.0}


In [11]:
#tokenizer = AutoTokenizer.from_pretrained(**ipa_tokenizer_name)
# Get phoneme vocabulary (excluding special tokens)
#known_phonemes = [p for p in tokenizer.get_vocab().keys() if p not in {"WORD_BOUNDARY", "UTT_BOUNDARY", "PAD", "UNK"}]

# Apply improved IPA preparation
#sample_texts = ipa_train_set["text"].apply(lambda x: prepare_ipa_text(x, known_phonemes))[:5].to_list()

#enc = tokenizer(sample_texts, padding=True, truncation=True, return_tensors="pt")

#print("Tokens:", enc["input_ids"])
#print("Decoded back:", [tokenizer.decode(ids) for ids in enc["input_ids"]])

#print('unk tokens count:', [tokenizer.decode(ids) for ids in enc["input_ids"]].count('UNK'))

# Compare Results

In [12]:
#print("✅ Baseline Accuracy:", metrics_base["eval_accuracy"])
print("✅ IPA Accuracy:", metrics_ipa["eval_accuracy"])

✅ IPA Accuracy: 0.5046


# Conclusions

